Pre-analysis of the data

In [1]:
print("a")

a


In [4]:
# Extract CSV's to PD's dataframes
import pandas as pd

finantial_impact = pd.read_csv("data/raw/financial_impact.csv")
master = pd.read_csv("data/raw/incidents_master.csv")
market_impact = pd.read_csv("data/raw/market_impact.csv")


We do need to do a general overview of the dataset we are working with. Since we are going to work as it was a real-world scenario, we will analyse the common columns on each of the csv.

In [27]:
# GENERAL VIEW
# row count, column count, join keys, date ranges, data source types
import pandas as pd

def general_view(df, df_others = []):
    print("GENERAL VIEW")
    df.info()
    print("==========")
    print("OTHER DATAFRAMES COMMON COLUMNS")

    if len(df_others) == 0:
        return
    
    relations_list = []
    global_common = []
    for col in df.columns:
        for i, df_object in enumerate(df_others):
            if len(relations_list) <= i:
                relations_list.append([i])
            
            if col in df_object.columns:
                relations_list[i].append(col)

    global_common = relations_list[0][1:]

    for df_object in relations_list:
        global_common = set(global_common).intersection(set(df_object[1:]))
        print(f"- Dataframe {df_object[0]}:", df_object[1:])
    
    print("Global common columns:", set(global_common))

general_view(master, [finantial_impact, market_impact])

GENERAL VIEW
<class 'pandas.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   incident_id               850 non-null    str    
 1   company_name              850 non-null    str    
 2   company_revenue_usd       850 non-null    float64
 3   country_hq                850 non-null    str    
 4   industry_primary          850 non-null    str    
 5   industry_secondary        153 non-null    str    
 6   employee_count            850 non-null    int64  
 7   is_public_company         850 non-null    bool   
 8   stock_ticker              412 non-null    str    
 9   incident_date             850 non-null    str    
 10  incident_date_estimated   850 non-null    bool   
 11  discovery_date            850 non-null    str    
 12  disclosure_date           850 non-null    str    
 13  attack_vector_primary     850 non-null    str    
 14  attack_v

We just identified a common identifier for all the dataframes, the column "incident_id". Now we would be able of trying to cross the data in case we need to for the common entries.

Now we will manage one of the most important problems in the pre-analysis stage. We need to get rid of the missing data, based on the their specific case.

In [37]:
master_nulls = master.isnull().mean().sort_values(ascending=False)
finantial_nulls = finantial_impact.isnull().mean().sort_values(ascending=False)
market_nulls = market_impact.isnull().mean().sort_values(ascending=False)

master_nulls[master_nulls > 0], finantial_nulls[finantial_nulls > 0], market_nulls[market_nulls > 0]


(review_flag                 0.917647
 industry_secondary          0.820000
 attack_vector_secondary     0.751765
 notes                       0.748235
 data_source_secondary       0.545882
 stock_ticker                0.515294
 downtime_hours              0.505882
 attributed_group            0.432941
 attribution_confidence      0.432941
 attack_chain                0.323529
 data_compromised_records    0.291765
 data_type                   0.291765
 dtype: float64,
 ransom_paid_usd         0.889460
 ransom_source           0.889460
 regulatory_fine_usd     0.830334
 ransom_demanded_usd     0.735219
 notes                   0.681234
 insurance_payout_usd    0.440874
 dtype: float64,
 notes                     0.743017
 days_to_price_recovery    0.100559
 dtype: float64)

As you can see, there are some columns with a high missingness-ratio. Even though, all the ones that could be considered as unrealable data (>60% missing), are under a regular collection ratio for its type of data. Some examples are high missingness on ransomwere related fields, since only ones that could fill that information are ransomware attack cases, 

In [ ]:
master.describe()